In [1]:
"""
Convert an NSI GeoPackage to a FAST-ready CSV.
Edit the paths/name in the Config cell, then run.
"""

'\nConvert an NSI GeoPackage to a FAST-ready CSV.\nEdit the paths/name in the Config cell, then run.\n'

## Imports

In [2]:
import geopandas as gpd
import fiona
from pathlib import Path
import getpass

# Get the current Windows user (e.g. "MeaganBrown") so paths work on any machine
USER = getpass.getuser()
USER_HOME = Path(f"C:/Users/{USER}")
print(f"Current user     : {USER}")

Current user     : MeaganBrown


## Config

In [3]:
### EDIT THESE
CLIP_NAME = "Solano_RSAP_Analysis_Area_v2.shp"
SITE_NAME  = "Solano"   # Used in output filename: NSI_<SITE_NAME>_FAST.csv

# Raw NSI Points
GPKG_PATH = rf"C:\Users\{USER}\Documents\GitHub\FAST\data\NSI_BayArea.gpkg"
gdf = gpd.read_file(GPKG_PATH, layer="NSI_BayArea")

CLIP_PATH = rf"C:\Users\{USER}\Documents\GitHub\FAST\Clip\{CLIP_NAME}"

# Output .csv
OUT_FOLDER = rf"C:\Users\{USER}\Documents\GitHub\FAST\UDF"
out_csv = Path(OUT_FOLDER) / f"NSI_{SITE_NAME}_FAST.csv"

In [4]:
# --- Read the GeoPackage ---
print("Layers in gpkg:", fiona.listlayers(GPKG_PATH))
gdf = gpd.read_file(GPKG_PATH)
print(f"Read {len(gdf):,} rows. CRS: {gdf.crs}")

# --- Reproject to WGS84 (lat/lon) so FAST gets degree coordinates ---
if gdf.crs and gdf.crs.to_epsg() != 4326:
    print(f"Reprojecting from {gdf.crs} to EPSG:4326...")
    gdf = gdf.to_crs(epsg=4326)

# --- clip to polygon, keeping only buildings inside it ---
if CLIP_PATH:
    print(f"\nClipping to polygon: {CLIP_PATH}")
    clip = gpd.read_file(CLIP_PATH)
    # Reproject the clip polygon to match the building data (WGS84)
    if clip.crs and clip.crs.to_epsg() != 4326:
        clip = clip.to_crs(epsg=4326)
    # Dissolve in case there are multiple polygons — we want one boundary
    clip_union = clip.unary_union
    before = len(gdf)
    gdf = gdf[gdf.geometry.within(clip_union)].copy()
    print(f"Clipped: kept {len(gdf):,} of {before:,} buildings within polygon")

# --- Rename NSI fields to FAST field names (case-insensitive match) ---
rename_map = {
    "occtype":    "Occ",                # Occupancy type (RES1, COM1, etc.)
    "val_struct": "Cost",               # Building replacement cost
    "num_story":  "NumStories",         # Number of stories
    "found_type": "FoundationType",     # Will be remapped from letters to 1-7 below
    "found_ht":   "FirstFloorHt",       # First floor height above grade (ft)
    "sqft":       "Area",               # Total building area (sq ft)
    "fd_id":      "UserDefinedFltyId",  # Unique building ID
}
lower_lookup  = {c.lower(): c for c in gdf.columns}
actual_rename = {lower_lookup[k.lower()]: v for k, v in rename_map.items() if k.lower() in lower_lookup}
gdf = gdf.rename(columns=actual_rename)
print("Renamed:", actual_rename)

# --- Pull Latitude / Longitude from the geometry (FAST needs them as explicit columns) ---
gdf["Longitude"] = gdf.geometry.x
gdf["Latitude"]  = gdf.geometry.y

# --- Convert NSI foundation letter codes (P/I/W/B/C/F/S) to FAST numeric codes (1-7) ---
foundation_map = {
    "P": 1,  # Pile
    "I": 2,  # Pier
    "W": 3,  # Solid Wall
    "B": 4,  # Basement
    "C": 5,  # Crawlspace
    "F": 6,  # Fill
    "S": 7,  # Slab
}
if "FoundationType" in gdf.columns:
    sample = gdf["FoundationType"].dropna().iloc[0] if gdf["FoundationType"].notna().any() else None
    if isinstance(sample, str):
        gdf["FoundationType"] = gdf["FoundationType"].str.upper().map(foundation_map)
        missing = gdf["FoundationType"].isna().sum()
        if missing:
            print(f"WARNING: {missing} rows had foundation codes not in the map — now blank")
    else:
        print("FoundationType already numeric — skipping letter map")

# --- Drop geometry column and write CSV ---
out_csv.parent.mkdir(parents=True, exist_ok=True)
gdf.drop(columns="geometry").to_csv(out_csv, index=False)
print(f"\nWrote {len(gdf):,} rows to {out_csv}")

Layers in gpkg: ['NSI_BayArea']
Read 2,241,196 rows. CRS: EPSG:4326

Clipping to polygon: C:\Users\MeaganBrown\Documents\GitHub\FAST\Clip\Solano_RSAP_Analysis_Area_v2.shp


C:\Users\MeaganBrown\AppData\Local\Temp\ipykernel_65296\478594585.py:19: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  clip_union = clip.unary_union


Clipped: kept 99,198 of 2,241,196 buildings within polygon
Renamed: {'occtype': 'Occ', 'val_struct': 'Cost', 'num_story': 'NumStories', 'found_type': 'FoundationType', 'found_ht': 'FirstFloorHt', 'sqft': 'Area', 'fd_id': 'UserDefinedFltyId'}

Wrote 99,198 rows to C:\Users\MeaganBrown\Documents\GitHub\FAST\UDF\NSI_Solano_FAST.csv
